In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, roc_auc_score

from lightgbm import LGBMRegressor

# ---------------------------
# 1. 데이터 로드
# ---------------------------
df = pd.read_csv(r"C:\Users\Admin\Documents\GitHub\Horse\data_preprocessing\merged_data.csv")

# ---------------------------
# 2. 타겟 생성
# ---------------------------
df["target"] = (df["rcRank"] == 1).astype(int)

# ---------------------------
# 3. 누수 제거 (중요)
# ---------------------------
drop_cols = [
    "rcRank",   # 정답
    "rcTime",   # 경기 결과
    "rcP1Odd","rcP2Odd","rcP3Odd","rcP4Odd","rcP5Odd","rcP6Odd","rcP8Odd"
]
df = df.drop(columns=drop_cols)

# ---------------------------
# 4. 날짜 처리
# ---------------------------
df["rcDate"] = pd.to_datetime(df["rcDate"], format="%Y%m%d", errors="coerce")

df["year"] = df["rcDate"].dt.year
df["month"] = df["rcDate"].dt.month

df = df.drop(columns=["rcDate"])

# ---------------------------
# 5. 범주형 인코딩
# ---------------------------
cat_cols = df.select_dtypes(include=["object"]).columns

for col in cat_cols:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

# ---------------------------
# 6. 분리
# ---------------------------
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ---------------------------
# 7. 모델
# ---------------------------
model = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    random_state=42
)

model.fit(X_train, y_train)

# ---------------------------
# 8. 예측
# ---------------------------
pred = model.predict(X_test)

# ---------------------------
# 9. 평가
# ---------------------------
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred)))
print("AUC:", roc_auc_score(y_test, pred))

# ---------------------------
# 10. 결과 확인
# ---------------------------
result = X_test.copy()
result["실제"] = y_test
result["예측값"] = pred

print(result.head())